# Создание ансамблей

Тот же зафиксированный препроцессинг, что и в classic_models_exploration/DNN_exploration. Базовые модели берём с лучшими гиперпараметрами, найденными в classic_models_exploration

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

from preprocessing import preprocess_data_advanced
from validation import cross_validate_model, evaluate_model, train_kfold_and_predict

train_data = pd.read_csv("data/train.csv")
test_data = pd.read_csv("data/test.csv")
test_passenger_ids = test_data['PassengerId']

train_data, artifacts = preprocess_data_advanced(train_data, is_train=True)
test_data = preprocess_data_advanced(test_data, is_train=False, artifacts=artifacts)

categorical_cols = ['Pclass', 'Embarked', 'Title']
train_data = pd.get_dummies(train_data, columns=categorical_cols)
test_data = pd.get_dummies(test_data, columns=categorical_cols)
test_data = test_data.reindex(columns=train_data.drop('Survived', axis=1).columns, fill_value=0)

scale_cols = ['Age', 'Fare', 'SibSp', 'Parch', 'TicketGroupSize']

scaler = StandardScaler()
train_data[scale_cols] = scaler.fit_transform(train_data[scale_cols])
test_data[scale_cols] = scaler.transform(test_data[scale_cols])

X_train = train_data.drop(['Survived'], axis=1)
y_train = train_data['Survived']
X_test = test_data

X_train.shape, X_test.shape

((891, 18), (418, 18))

Базовые модели с лучшими гиперпараметрами из classic_models_exploration. Оценим каждую по отдельности через CV, чтобы было с чем сравнивать ансамбли

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

results = {}

base_models = {
    'logreg': LogisticRegression(solver='saga', max_iter=5000, random_state=42, C=1, l1_ratio=0.75),
    'random_forest': RandomForestClassifier(
        n_estimators=300, max_depth=10, max_features='sqrt', criterion='entropy', min_samples_leaf=1,
        random_state=42, n_jobs=-1
    ),
    'xgboost': XGBClassifier(
        colsample_bytree=0.7, learning_rate=0.05, max_depth=3, n_estimators=300, subsample=1.0, random_state=42
    ),
    'lightgbm': LGBMClassifier(
        colsample_bytree=0.7, learning_rate=0.01, max_depth=-1, n_estimators=300, subsample=0.7,
        random_state=42, verbosity=-1
    ),
    'catboost': CatBoostClassifier(
        depth=4, iterations=300, l2_leaf_reg=1, learning_rate=0.03,
        random_state=42, verbose=False, allow_writing_files=False, thread_count=-1
    ),
}

for name, model in base_models.items():
    evaluate_model(name, model, X_train, y_train, results)

logreg: 0.83387 +- 0.01288
random_forest: 0.83948 +- 0.01287
xgboost: 0.84622 +- 0.02192
lightgbm: 0.84510 +- 0.01845
catboost: 0.84398 +- 0.00581


Пункт 1: усреднение = soft voting (усредняем предсказанные вероятности всех 5 базовых моделей и берём класс с большей средней вероятностью)

In [3]:
from sklearn.ensemble import VotingClassifier

estimators = list(base_models.items())

voting_soft = VotingClassifier(estimators=estimators, voting='soft', n_jobs=-1)
evaluate_model('averaging_soft_voting', voting_soft, X_train, y_train, results)

averaging_soft_voting: 0.84511 +- 0.01056


array([0.8547486 , 0.85393258, 0.8258427 , 0.84831461, 0.84269663])

Пункт 2: voting = hard voting (голосование по предсказанным классам, побеждает большинство из 5 моделей, а не усреднённая вероятность)

In [4]:
voting_hard = VotingClassifier(estimators=estimators, voting='hard', n_jobs=-1)
evaluate_model('voting_hard', voting_hard, X_train, y_train, results)

voting_hard: 0.84735 +- 0.00987


array([0.8603352 , 0.85393258, 0.83146067, 0.84831461, 0.84269663])

Hard voting (0.847) обошёл все 5 базовых моделей по отдельности, включая лучший XGBoost (0.846), и разброс заметно ниже (std 0.010 против 0.022 у XGBoost). Soft voting (0.845) слабее hard voting и на уровне LightGBM - вероятно, слабый logreg (0.834) сильнее тянет вниз усреднённую вероятность, чем один голос из пяти при hard voting

Пункт 3: stacking через линейную регрессию (обычный LogisticRegression без регуляризации) и Ridge (RidgeClassifier). Внутренний cv=3 у StackingClassifier (генерирует out-of-fold предсказания базовых моделей для мета-модели), уменьшен с дефолтных 5 ради скорости - каждый базовый бустинг и так обучается многократно

In [5]:
from sklearn.ensemble import StackingClassifier

# C=np.inf = без регуляризации в sklearn 1.9 (penalty устарел, см. classic_models_exploration)
stacking_logreg = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(C=np.inf, max_iter=5000, random_state=42),
    cv=3, n_jobs=-1
)
evaluate_model('stacking_logreg', stacking_logreg, X_train, y_train, results)

stacking_logreg: 0.84061 +- 0.01128


array([0.8603352 , 0.83707865, 0.8258427 , 0.84269663, 0.83707865])

Stacking с логрегом (0.841) хуже hard voting (0.847) и даже хуже одного XGBoost (0.846). На 891 строке мета-модели не хватает данных, чтобы аккуратно выучить веса для 5 базовых моделей поверх cv=3 out-of-fold предсказаний - переобучается на уровне мета-слоя

In [6]:
from sklearn.linear_model import RidgeClassifier

stacking_ridge = StackingClassifier(
    estimators=estimators,
    final_estimator=RidgeClassifier(random_state=42),
    cv=3, n_jobs=-1
)
evaluate_model('stacking_ridge', stacking_ridge, X_train, y_train, results)

stacking_ridge: 0.84286 +- 0.00960


array([0.8547486 , 0.84831461, 0.8258427 , 0.84269663, 0.84269663])

Ridge (0.843) чуть лучше логрега без регуляризации (0.841) - регуляризация мета-модели немного помогает не переобучиться на 5 сильно коррелированных базовых предсказаниях, но всё ещё хуже и hard voting (0.847), и одного XGBoost (0.846). Stacking на таком маленьком датасете и таком небольшом количестве непохожих базовых моделей не оправдывает себя

Сведём все результаты в таблицу

In [7]:
summary = pd.DataFrame({
    'mean': {name: scores.mean() for name, scores in results.items()},
    'std': {name: scores.std() for name, scores in results.items()},
}).sort_values('mean', ascending=False)

summary

,mean,std
voting_hard,0.847348,0.009869
xgboost,0.846218,0.021921
averaging_soft_voting,0.845107,0.010565
lightgbm,0.845101,0.018453
catboost,0.843983,0.005806
stacking_ridge,0.842860,0.009599
stacking_logreg,0.840606,0.011282
random_forest,0.839483,0.012865
logreg,0.833871,0.012876


Итог: **hard voting (0.847) - новый общий лидер проекта**, обошёл лучшую одиночную модель (XGBoost, 0.846) и почти вдвое снизил разброс (std 0.010 против 0.022). Soft voting и обе версии stacking не дотянули даже до одиночного XGBoost - усложнение ансамбля не помогает без более разнообразных и сильных базовых моделей. Простое голосование по 5 разным алгоритмам (логрег, RF, XGBoost, LightGBM, CatBoost) оказалось надёжнее, чем попытка выучить веса или вероятности поверх них

Попробуем добавить в ансамбль DNN (лучшая конфигурация из DNN_exploration: EmbeddingMLP, embedding_dim=2, hidden=16, Tanh, Adam lr=1e-3, честный CV 0.842). VotingClassifier/StackingClassifier ждут sklearn-совместимый интерфейс (fit/predict/predict_proba, get_params/set_params) и единую матрицу X, а EmbeddingMLP принимает отдельно категориальные индексы и числа. Обёртка берёт ту же one-hot X_train, что и остальные модели, и сама разбирает её обратно на индексы категорий (argmax по one-hot колонкам каждой группы) и числовой блок

In [8]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.base import BaseEstimator, ClassifierMixin


class EmbeddingMLP(nn.Module):
    def __init__(self, cat_cardinalities, embedding_dims, num_numeric, hidden_dim=16, activation=nn.Tanh):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(cardinality, dim) for cardinality, dim in zip(cat_cardinalities, embedding_dims)
        ])
        self.fc1 = nn.Linear(sum(embedding_dims) + num_numeric, hidden_dim)
        self.activation = activation()
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, x_cat, x_num):
        embedded = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.cat(embedded + [x_num], dim=1)
        return self.fc2(self.activation(self.fc1(x)))


# группы one-hot колонок, которые argmax'ом сворачиваются обратно в индекс категории
categorical_groups = {
    'Pclass': ['Pclass_1', 'Pclass_2', 'Pclass_3'],
    'Embarked': ['Embarked_C', 'Embarked_Q', 'Embarked_S'],
    'Title': ['Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare'],
}
numeric_cols_dnn = ['Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'TicketGroupSize', 'HasCabin']


class EmbeddingMLPClassifier(ClassifierMixin, BaseEstimator):
    """sklearn-совместимая обёртка над EmbeddingMLP, чтобы её можно было класть в VotingClassifier/StackingClassifier.
    ClassifierMixin обязательно первым в списке родителей - в sklearn 1.9 иначе is_classifier() возвращает False
    (ClassifierMixin.__sklearn_tags__ должен отработать раньше BaseEstimator в MRO)"""

    def __init__(self, embedding_dim=2, hidden_dim=16, lr=1e-3, n_epochs=50, batch_size=32, random_state=42):
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.lr = lr
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.random_state = random_state

    def _split_inputs(self, X):
        x_cat_cols = [X[cols].values.argmax(axis=1) for cols in categorical_groups.values()]
        x_cat = np.stack(x_cat_cols, axis=1)
        x_num = X[numeric_cols_dnn].values
        return torch.tensor(x_cat, dtype=torch.long), torch.tensor(x_num, dtype=torch.float32)

    def fit(self, X, y):
        torch.manual_seed(self.random_state)
        x_cat, x_num = self._split_inputs(X)
        y_arr = y.values if hasattr(y, 'values') else np.asarray(y)
        y_t = torch.tensor(y_arr, dtype=torch.float32).unsqueeze(1)

        cardinalities = [len(cols) for cols in categorical_groups.values()]
        self.model_ = EmbeddingMLP(
            cardinalities, [self.embedding_dim] * len(cardinalities), len(numeric_cols_dnn),
            self.hidden_dim, nn.Tanh
        )
        optimizer = torch.optim.Adam(self.model_.parameters(), lr=self.lr)
        loss_fn = nn.BCEWithLogitsLoss()
        loader = DataLoader(TensorDataset(x_cat, x_num, y_t), batch_size=self.batch_size, shuffle=True)

        self.model_.train()
        for epoch in range(self.n_epochs):
            for xc, xn, yb in loader:
                optimizer.zero_grad()
                loss = loss_fn(self.model_(xc, xn), yb)
                loss.backward()
                optimizer.step()

        self.classes_ = np.array([0, 1])
        return self

    def predict_proba(self, X):
        x_cat, x_num = self._split_inputs(X)
        self.model_.eval()
        with torch.no_grad():
            proba_1 = torch.sigmoid(self.model_(x_cat, x_num)).numpy().ravel()
        return np.column_stack([1 - proba_1, proba_1])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

In [9]:
dnn_model = EmbeddingMLPClassifier()
evaluate_model('dnn_embedding', dnn_model, X_train, y_train, results)

estimators_with_dnn = estimators + [('dnn', dnn_model)]

dnn_embedding: 0.84061 +- 0.01327


Прогоним hard и soft voting с добавленной DNN (6 моделей вместо 5)

In [10]:
voting_hard_with_dnn = VotingClassifier(estimators=estimators_with_dnn, voting='hard', n_jobs=-1)
evaluate_model('voting_hard_with_dnn', voting_hard_with_dnn, X_train, y_train, results)

voting_soft_with_dnn = VotingClassifier(estimators=estimators_with_dnn, voting='soft', n_jobs=-1)
evaluate_model('averaging_soft_voting_with_dnn', voting_soft_with_dnn, X_train, y_train, results)

voting_hard_with_dnn: 0.84398 +- 0.01265
averaging_soft_voting_with_dnn: 0.84622 +- 0.01175


array([0.8603352 , 0.84831461, 0.8258427 , 0.85393258, 0.84269663])

DNN добавлять не стоило: hard voting с DNN (0.844) хуже hard voting без неё (0.847), soft voting с DNN (0.846) чуть лучше soft voting без неё (0.845), но всё равно хуже hard voting без DNN. Сама DNN (0.841) слабее всех бустингов и logreg - её "голос" в hard voting только размывает большинство сильных моделей, а её менее уверенные вероятности в soft voting не докручивают результат выше уровня одного XGBoost. Лучший ансамбль в проекте - hard voting на исходных 5 моделях (0.847), без DNN

Обновлённая итоговая таблица со всеми экспериментами, включая DNN

In [11]:
summary_v2 = pd.DataFrame({
    'mean': {name: scores.mean() for name, scores in results.items()},
    'std': {name: scores.std() for name, scores in results.items()},
}).sort_values('mean', ascending=False)

summary_v2

,mean,std
voting_hard,0.847348,0.009869
averaging_soft_voting_with_dnn,0.846224,0.011754
xgboost,0.846218,0.021921
averaging_soft_voting,0.845107,0.010565
lightgbm,0.845101,0.018453
catboost,0.843983,0.005806
voting_hard_with_dnn,0.843983,0.012647
stacking_ridge,0.842860,0.009599
dnn_embedding,0.840613,0.013274
stacking_logreg,0.840606,0.011282


Финал проекта: **hard voting на 5 классических моделях/бустингах (0.847, без DNN) - лучший результат во всём проекте**. DNN интересна сама по себе, но как участник ансамбля не пригодилась - она слабее остальных базовых моделей и только размывает голосование. Ансамбли и feature engineering раунд 2 подтверждают вывод из classic_models_exploration: дальнейший рост качества упирается не в архитектуру моделей, а в потолок сигнала, который есть в этих 891 строке и текущем наборе фичей